In [3]:
import math
import random
from collections import deque
import pygame

pygame.mixer.pre_init(22050, -16, 1, 512)
pygame.init()

WIDTH, HEIGHT = 900, 600
FPS = 60
screen = pygame.display.set_mode((WIDTH, HEIGHT))
canvas = pygame.Surface((WIDTH, HEIGHT))
pygame.display.set_caption("Neon Pong")
clock = pygame.time.Clock()

title_font = pygame.font.SysFont("arial", 120, bold=True)
huge_font = pygame.font.SysFont("arial", 170, bold=True)
big_font = pygame.font.SysFont("arial", 54, bold=True)
font = pygame.font.SysFont("arial", 30)
small_font = pygame.font.SysFont("arial", 21)

WHITE = (255, 255, 255)
GOLD = (255, 215, 0)
GREY = (150, 155, 190)
GREEN = (90, 225, 130)
YELLOW = (255, 205, 70)
RED = (255, 95, 95)
P1_COLOR = (0, 220, 255)
P2_COLOR = (255, 80, 160)
BG_TOP = (12, 14, 40)
BG_BOTTOM = (42, 16, 66)
BG_MID = (27, 15, 53)

PADDLE_W, PADDLE_H = 16, 100
PADDLE_SPEED = 9
LEFT_X = 40
RIGHT_X = WIDTH - 40 - PADDLE_W
DIFFICULTIES = ["Easy", "Medium", "Hard"]
MIN_TARGET, MAX_TARGET = 1, 21


def mix(c1, c2, t):
    t = max(0, min(1, t))
    return tuple(int(c1[i] + (c2[i] - c1[i]) * t) for i in range(3))


def draw_text(text, f, color, center, scale=1.0):
    if not text:
        return
    img = f.render(text, True, color)
    if scale != 1.0:
        img = pygame.transform.rotozoom(img, 0, scale)
    canvas.blit(img, img.get_rect(center=center))


def make_background():
    surf = pygame.Surface((WIDTH, HEIGHT))
    for y in range(HEIGHT):
        color = mix(BG_TOP, BG_BOTTOM, y / HEIGHT)
        pygame.draw.line(surf, color, (0, y), (WIDTH, y))
    return surf

def make_circle_glow(radius, color, layers=10):   
    surf = pygame.Surface((radius * 2, radius * 2), pygame.SRCALPHA)   
    for i in range(layers, 0, -1):          
        r = int(radius * i / layers)       
        alpha = int(6 + 80 * (1 - i / layers) ** 1.5)   
        pygame.draw.circle(surf, (*color, alpha), (radius, radius), r)
    return surf

def make_rect_glow(w, h, color, layers=8, step=3): 
    pad = layers * step                     
    surf = pygame.Surface((w + 2 * pad, h + 2 * pad), pygame.SRCALPHA)  
    for i in range(layers, 0, -1):          
        grow = i * step                     
        rect = pygame.Rect(pad - grow, pad - grow, w + 2 * grow, h + 2 * grow)   
        alpha = int(70 * (1 - i / (layers + 1)))   
        pygame.draw.rect(surf, (*color, alpha), rect, border_radius=8 + grow)   
    return surf, pad  

background = make_background()             
ball_glow = make_circle_glow(34, (130, 170, 255))   
dim = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)   
dim.fill((8, 8, 24, 175))                  

def make_sound(notes, volume=0.2):      
    sample_rate = 22050                   
    data = bytearray()                    
    for freq, ms in notes:                  
        n_samples = int(sample_rate * ms / 1000)   
        for i in range(n_samples):        
            attack = min(1, i / 200)      
            decay = math.exp(-4 * i / n_samples)  
            wave = math.sin(2 * math.pi * freq * i / sample_rate) 
            value = int(volume * 32767 * attack * decay * wave)    
            data += value.to_bytes(2, "little", signed=True)      
    return pygame.mixer.Sound(buffer=bytes(data))   

try:                                        
    snd_move = make_sound([(600, 45)], 0.10)                                  
    snd_select = make_sound([(700, 60), (930, 110)], 0.14)                    
    snd_hit = make_sound([(480, 90)], 0.20)                                   
    snd_wall = make_sound([(330, 60)], 0.12)                                  
    snd_score = make_sound([(523, 90), (392, 220)], 0.18)                     
    snd_win = make_sound([(523, 120), (659, 120), (784, 120), (1047, 380)], 0.2)   
    snd_count = make_sound([(500, 110)], 0.14)                               
    snd_go = make_sound([(800, 220)], 0.16)                                   
    sound_on = True                        
except Exception:                         
    sound_on = False   
    snd_move = snd_select = snd_hit = snd_wall = None     
    snd_score = snd_win = snd_count = snd_go = None


pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.8)
